# 3.2 Deploy agents with Anthropic MCP

# Problem setup

The goal of this section is to build an AI agent that can analyze protein data and generate drug discovery hypotheses step by step.

When you understand the basic workflow, you will be able to design and build your own agents.

Let's start with installing dependencies and loading API keys below. We will be using the Anthropic MCP in this example, so make sure you have an Anthropic API key.

In [ ]:
!pip install anthropic biopython

In [ ]:
import os
from dotenv import load_dotenv

def get_api_key(provider: str = "anthropic"):
    """Load Anthropic API key from Colab secrets or environment variable."""
    if provider == "anthropic":
        load_key = "ANTHROPIC_API_KEY"
    elif provider == "openai":
        load_key = "OPENAI_API_KEY"
    else:
        raise ValueError(f"Invalid provider: {provider} only anthropic and openai are supported")
    try:
        from google.colab import userdata
        return userdata.get(load_key)
    except ImportError:
        # Not in Colab — fall back to environment variable
        load_dotenv() 
        api_key = os.environ.get(load_key)
        if not api_key:
            raise ValueError(
                "API key not found. Please set the ANTHROPIC_API_KEY or OPENAI_API_KEY environment variable.\n"
                "You can do this by running the following in your terminal. Example:\n"
                "  export ANTHROPIC_API_KEY='your-key-here'\n"
                "Or add it to a .env file in your project root."
            )
        return api_key


## Step 1: import libraries and set up the LLM client

In [ ]:
import anthropic
from Bio.SeqUtils.ProtParam import ProteinAnalysis 

# Connect to the Claude AI model.
# Make sure you have set your API key as an environment variable:
# export ANTHROPIC_API_KEY="sk-ant-..."
client = anthropic.Anthropic(api_key=get_api_key())

print("✅ Libraries loaded and client ready.")

## Step 2: define the tools
As we discussed previously, tools are Python functions our agent can choose to call. 

We give each tool a name, a description (so the AI understands what it does), and a list of inputs it expects.
In this example we'll build **two tools**:
- `analyze_protein_sequence` — computes basic biophysical properties
- `summarize_protein_role`   — asks the AI to summarize biological context


In [ ]:
# ── TOOL 1: Analyze a protein sequence ──────────────────────────────────────

def analyze_protein_sequence(sequence: str) -> dict:
    """
    Takes a protein sequence (one-letter amino acid codes, e.g. 'MKTIIALSYIFCLVFA...')
    and returns basic biophysical properties using BioPython.
    """
    sequence = sequence.upper().strip()
    analysis = ProteinAnalysis(sequence)

    results = {
        "length":              len(sequence),
        "molecular_weight_Da": round(analysis.molecular_weight(), 2),
        "isoelectric_point":   round(analysis.isoelectric_point(), 2),
        "instability_index":   round(analysis.instability_index(), 2),
        "gravy_score":         round(analysis.gravy(), 3),   # hydrophobicity
        "amino_acid_percent":  {
            aa: round(pct, 1)
            for aa, pct in analysis.get_amino_acids_percent().items()
            if pct > 0   # only show amino acids actually present
        },
    }

    # Interpret some values for the non-expert
    results["is_stable"]     = results["instability_index"] < 40
    results["is_hydrophilic"] = results["gravy_score"] < 0

    return results


# ── TOOL 2: Summarize protein biological role ────────────────────────────────

def summarize_protein_role(protein_name: str, organism: str) -> dict:
    """
    Uses a separate Claude call to look up and summarize the biological
    role of a protein from its training knowledge.
    """
    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=400,
        messages=[{
            "role": "user",
            "content": (
                f"Provide a concise 3–4 sentence summary of the protein '{protein_name}' "
                f"in {organism}. Cover: (1) its biological function, "
                f"(2) which disease or condition it is associated with, "
                f"(3) why it is considered a drug target. Be factual and concise."
            )
        }]
    )
    return {"summary": response.content[0].text}

Now let's package these as **tool definitions**. This is a structured description the AI model can read to understand what tools are available.

In [ ]:
tools = [
    {
        "name": "analyze_protein_sequence",
        "description": (
            "Analyzes a raw protein amino acid sequence and returns biophysical "
            "properties: length, molecular weight, isoelectric point, instability "
            "index, GRAVY hydrophobicity score, and amino acid composition."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "sequence": {
                    "type": "string",
                    "description": "The protein sequence in one-letter amino acid code, e.g. 'MKTII...'"
                }
            },
            "required": ["sequence"]
        }
    },
    {
        "name": "summarize_protein_role",
        "description": (
            "Summarizes the known biological role, disease association, and "
            "drug-target relevance of a named protein in a given organism."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "protein_name": {
                    "type": "string",
                    "description": "Common name or gene symbol of the protein, e.g. 'EGFR' or 'p53'"
                },
                "organism": {
                    "type": "string",
                    "description": "The organism, e.g. 'human', 'mouse', 'E. coli'"
                }
            },
            "required": ["protein_name", "organism"]
        }
    }
]

print("✅ Tools defined.")


## Step 3 – Build the agent loop